# Tennis domain adaptation experiments

This Colab notebook is an orchestration layer for the repository scripts. It does not duplicate model loading, generation, scoring, or training logic. Use a GPU runtime for any model evaluation, and start with smoke-mode commands before enabling full runs.

## 1. Runtime setup

Recommended Colab runtime: GPU. The cells below check GPU visibility, mount Google Drive, set the project path, optionally clone the repository, and move into the repo root.

In [ ]:
from shutil import which`n
import subprocess`n
`n
if which("nvidia-smi"):`n
    subprocess.run(["nvidia-smi"], check=False)`n
else:`n
    print("nvidia-smi not found; continuing without a visible NVIDIA GPU.")

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Google Drive mount skipped: {exc}')

In [ ]:
from pathlib import Path
import subprocess

# Edit this path for your Drive layout or cloned checkout.
PROJECT_ROOT = Path('/content/drive/MyDrive/tiser_temporal_reasoning_extension')

# Optional. Set to your GitHub repo URL if PROJECT_ROOT does not already exist.
GITHUB_REPO_URL = ''

if not PROJECT_ROOT.exists():
    if GITHUB_REPO_URL:
        PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(f'git clone "{GITHUB_REPO_URL}" "{PROJECT_ROOT}"', shell=True, check=True)
    else:
        raise FileNotFoundError(
            f'{PROJECT_ROOT} does not exist. Set PROJECT_ROOT to your repo path or set GITHUB_REPO_URL.'
        )

print(PROJECT_ROOT)

In [ ]:
%cd {PROJECT_ROOT}

## 2. Install dependencies

In [ ]:
import os`n
import shlex`n
import subprocess`n
import sys`n
import textwrap`n
`n
def normalize_command(command):`n
    command = textwrap.dedent(command).strip()`n
    command = command.replace(" \\\n", " ")`n
    return " ".join(line.strip() for line in command.splitlines() if line.strip())`n
`n
def run_shell(command, *, enabled=True):`n
    command = normalize_command(command)`n
    if not enabled:`n
        print("[skip] Stage disabled. Command retained for reproducibility:")`n
        print(command)`n
        return`n
    print("[run]")`n
    print(command)`n
    args = shlex.split(command, posix=True)`n
    if args and args[0] == "python":`n
        args[0] = sys.executable`n
    subprocess.run(args, check=True)

In [ ]:
import importlib.util
import subprocess
import sys

runtime_dependencies = {
    'accelerate': 'accelerate',
    'bitsandbytes': 'bitsandbytes',
    'datasets': 'datasets',
    'peft': 'peft',
    'torch': 'torch',
    'transformers': 'transformers',
    'trl': 'trl',
    'yaml': 'pyyaml',
}
missing = [package for module, package in runtime_dependencies.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing missing runtime dependencies:', missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
else:
    print('All checked runtime dependencies are importable.')

In [ ]:
import platform
import torch

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))

## 3. Configuration flags

Full 7B evaluations and training are disabled by default. Enable only the stages you intend to run.

In [ ]:
RUN_SMOKE = True
RUN_BASE_QWEN = False
RUN_ORIGINAL_TISER = False
RUN_TENNIS_ONLY_EVAL = False
RUN_MIXED_REPLAY_EVAL = False
RUN_AGGREGATION = True
RUN_TRAINING = False

LIMIT = 5
BATCH_SIZE = 1
MAX_NEW_TOKENS = 256

## 4. Path configuration

In [ ]:
TENNIS_TEST = 'data/tennis/tennis_test.json'
TENNIS_TRAIN_TRACED = 'data/tennis/tennis_train_traced.json'

ORIGINAL_TISER_ADAPTER = 'model/tiser_qwen7b_full/adapter'
TENNIS_ONLY_ADAPTER = 'model/tennis_only_qwen7b/adapter'
MIXED_REPLAY_ADAPTER = 'model/mixed_tennis_tiser_replay_qwen7b/adapter'

CONFIG = 'config/config_tennis.yaml'
SMOKE_CONFIG = 'config/config_tennis_smoke.yaml'

RESULTS_DIR = 'results/tennis_domain_adaptation'

In [ ]:
import subprocess
import textwrap

def run_shell(command, *, enabled=True):
    command = textwrap.dedent(command).strip()
    if not enabled:
        print('[skip] Stage disabled. Command retained for reproducibility:')
        print(command)
        return
    print('[run]')
    print(command)
    subprocess.run(command, shell=True, check=True)

## 5. Preflight checks

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

def preflight_row(label, path, *, kind='file', required=True, enabled=True):
    p = Path(path)
    exists = p.is_dir() if kind == 'dir' else p.exists()
    if exists:
        status = 'PASS'
        note = 'found'
    elif enabled and required:
        status = 'FAIL'
        note = 'missing and required for enabled stage'
    else:
        status = 'WARN'
        note = 'missing; stage is disabled or script is optional'
    return {'status': status, 'item': label, 'path': str(p), 'note': note}

checks = [
    preflight_row('tennis test file', TENNIS_TEST),
    preflight_row('tennis train traced file', TENNIS_TRAIN_TRACED, required=False, enabled=RUN_TRAINING),
    preflight_row('config', CONFIG),
    preflight_row('smoke config', SMOKE_CONFIG),
    preflight_row('original TISER adapter', ORIGINAL_TISER_ADAPTER, kind='dir', enabled=RUN_ORIGINAL_TISER),
    preflight_row('tennis-only adapter', TENNIS_ONLY_ADAPTER, kind='dir', enabled=RUN_TENNIS_ONLY_EVAL),
    preflight_row('mixed replay adapter', MIXED_REPLAY_ADAPTER, kind='dir', enabled=RUN_MIXED_REPLAY_EVAL),
    preflight_row('evaluate_tennis.py', 'scripts/tennis/evaluate_tennis.py'),
    preflight_row('compare_adapters.py', 'scripts/tennis/compare_adapters.py'),
    preflight_row('run_experiment_plan.py', 'scripts/tennis/run_experiment_plan.py', required=False, enabled=False),
    preflight_row('aggregate_tennis_results.py', 'scripts/tennis/aggregate_tennis_results.py', required=False, enabled=False),
    preflight_row('train_tennis.py', 'scripts/tennis/train_tennis.py', required=False, enabled=RUN_TRAINING),
]

header = '| Status | Item | Path | Note |\n|---|---|---|---|'
body = '\n'.join(f"| {row['status']} | {row['item']} | `{row['path']}` | {row['note']} |" for row in checks)
display(Markdown(header + '\n' + body))

failures = [row for row in checks if row['status'] == 'FAIL']
if failures:
    raise FileNotFoundError('Preflight failed for: ' + ', '.join(row['item'] for row in failures))

## 6. Smoke evaluation

In [ ]:
run_shell(f'''
python scripts/tennis/evaluate_tennis.py \
  --config "{CONFIG}" \
  --test-file "{TENNIS_TEST}" \
  --condition base_qwen_smoke \
  --no-adapter \
  --limit {LIMIT} \
  --batch-size 1 \
  --max-new-tokens 256 \
  --output-dir "{RESULTS_DIR}/scored/base_qwen_smoke"
''', enabled=RUN_SMOKE)

## 7. Full baseline evaluations

In [4]:
run_shell(f'''
python scripts/tennis/evaluate_tennis.py \
  --config "{CONFIG}" \
  --test-file "{TENNIS_TEST}" \
  --condition base_qwen \
  --no-adapter \
  --batch-size {BATCH_SIZE} \
  --max-new-tokens {MAX_NEW_TOKENS} \
  --output-dir "{RESULTS_DIR}/scored/base_qwen"
''', enabled=RUN_BASE_QWEN)

NameError: name 'run_shell' is not defined

In [ ]:
run_shell(f'''
python scripts/tennis/evaluate_tennis.py \
  --config "{CONFIG}" \
  --test-file "{TENNIS_TEST}" \
  --adapter-dir "{ORIGINAL_TISER_ADAPTER}" \
  --condition original_tiser \
  --batch-size {BATCH_SIZE} \
  --max-new-tokens {MAX_NEW_TOKENS} \
  --output-dir "{RESULTS_DIR}/scored/original_tiser"
''', enabled=RUN_ORIGINAL_TISER)

## 8. Tennis adapter evaluations

In [ ]:
run_shell(f'''
python scripts/tennis/evaluate_tennis.py \
  --config "{CONFIG}" \
  --test-file "{TENNIS_TEST}" \
  --adapter-dir "{TENNIS_ONLY_ADAPTER}" \
  --condition tennis_only \
  --batch-size {BATCH_SIZE} \
  --max-new-tokens {MAX_NEW_TOKENS} \
  --output-dir "{RESULTS_DIR}/scored/tennis_only"
''', enabled=RUN_TENNIS_ONLY_EVAL)

In [ ]:
run_shell(f'''
python scripts/tennis/evaluate_tennis.py \
  --config "{CONFIG}" \
  --test-file "{TENNIS_TEST}" \
  --adapter-dir "{MIXED_REPLAY_ADAPTER}" \
  --condition mixed_tennis_tiser_replay \
  --batch-size {BATCH_SIZE} \
  --max-new-tokens {MAX_NEW_TOKENS} \
  --output-dir "{RESULTS_DIR}/scored/mixed_tennis_tiser_replay"
''', enabled=RUN_MIXED_REPLAY_EVAL)

## 9. Experiment plan runner

This writes `results/tennis_domain_adaptation/comparisons/run_tennis_experiments.sh` in dry-run mode when the plan script exists. It does not execute the generated full experiment script.

In [ ]:
if Path('scripts/tennis/run_experiment_plan.py').exists():
    run_shell(f'''
python scripts/tennis/run_experiment_plan.py \
  --config "{CONFIG}" \
  --tennis-test "{TENNIS_TEST}" \
  --original-tiser-adapter "{ORIGINAL_TISER_ADAPTER}" \
  --tennis-adapter "{TENNIS_ONLY_ADAPTER}" \
  --mixed-adapter "{MIXED_REPLAY_ADAPTER}" \
  --limit {LIMIT}
''', enabled=True)
else:
    print('scripts/tennis/run_experiment_plan.py not found; skipping dry-run plan generation.')

In [ ]:
run_shell(f'''
python scripts/tennis/train_tennis.py \
  --config "{CONFIG}" \
  --train-file "{TENNIS_TRAIN_TRACED}" \
  --run-name tennis_only_qwen7b
''', enabled=RUN_TRAINING)

## 10. Aggregation

In [ ]:
run_shell(f'''
python scripts/tennis/compare_adapters.py \
  --results-dir "{RESULTS_DIR}"
''', enabled=RUN_AGGREGATION)

if Path('scripts/tennis/aggregate_tennis_results.py').exists():
    run_shell(f'''
python scripts/tennis/aggregate_tennis_results.py \
  --results-dir "{RESULTS_DIR}"
''', enabled=RUN_AGGREGATION)
else:
    print('scripts/tennis/aggregate_tennis_results.py not found; skipping optional aggregation.')

## 11. Display result summaries

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

summary_files = [
    Path(RESULTS_DIR) / 'comparisons/adapter_comparison.md',
    Path(RESULTS_DIR) / 'comparisons/final_results_table.md',
    Path(RESULTS_DIR) / 'comparisons/category_analysis.md',
    Path(RESULTS_DIR) / 'comparisons/forgetting_analysis.md',
]

for path in summary_files:
    if path.exists():
        display(Markdown(f'### {path.as_posix()}'))
        display(Markdown(path.read_text(encoding='utf-8')))
    else:
        print(f'[missing] {path.as_posix()}')

## 12. Save artifacts

In [ ]:
from pathlib import Path
import shutil

COPY_ZIP_TO_DRIVE = False
DRIVE_ZIP_TARGET = Path('/content/drive/MyDrive/tennis_domain_adaptation_results.zip')

results_path = Path(RESULTS_DIR)
if not results_path.exists():
    print(f'Results directory does not exist yet: {results_path}')
else:
    archive_path = shutil.make_archive(
        'tennis_domain_adaptation_results',
        'zip',
        root_dir=results_path.parent,
        base_dir=results_path.name,
    )
    print(f'Wrote {archive_path}')
    if COPY_ZIP_TO_DRIVE:
        DRIVE_ZIP_TARGET.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(archive_path, DRIVE_ZIP_TARGET)
        print(f'Copied to {DRIVE_ZIP_TARGET}')

## 13. Safety notes

- Do not run full 7B evaluations unless GPU memory is sufficient.
- Start with `--limit 5`.
- The smoke config using Qwen2.5-0.5B is only for base-model smoke checks, not for 7B adapters.
- Adapter model must match Qwen/Qwen2.5-7B-Instruct.
- Training should only be run after `tennis_train_traced.json` has validated TISER outputs.
- Keep checkpoints and large generated outputs out of git; final experiment summaries should live under `results/tennis_domain_adaptation/`.